In [3]:
import os
os.chdir("/Users/adnan/Desktop/healthcare-bi-portfolio")

import pandas as pd
import sqlite3
import os

df = pd.read_csv("data/raw/complications_deaths.csv")
print(f"Raw shape: {df.shape}")

df = df.drop(columns=["Footnote", "Telephone Number", "Address"])

df.columns = [
    "facility_id", "facility_name", "city", "state", "zip_code",
    "county", "measure_id", "measure_name", "compared_to_national",
    "denominator", "score", "lower_estimate", "higher_estimate",
    "start_date", "end_date"
]

numeric_cols = ["denominator", "score", "lower_estimate", "higher_estimate"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["start_date"] = pd.to_datetime(df["start_date"], format="%m/%d/%Y")
df["end_date"]   = pd.to_datetime(df["end_date"],   format="%m/%d/%Y")

df = df[df["end_date"] == df["end_date"].max()]
print(f"After filtering to latest period: {df.shape}")

df = df.dropna(subset=["score"])
print(f"After dropping null scores: {df.shape}")

os.makedirs("data/clean", exist_ok=True)
df.to_csv("data/clean/hospitals_clean.csv", index=False)
print("Saved: data/clean/hospitals_clean.csv")

conn = sqlite3.connect("data/healthcare.db")
df.to_sql("hospital_measures", conn, if_exists="replace", index=False)
conn.close()
print("Loaded into SQLite: data/healthcare.db")

Raw shape: (95780, 18)
After filtering to latest period: (90991, 15)
After dropping null scores: (50478, 15)
Saved: data/clean/hospitals_clean.csv
Loaded into SQLite: data/healthcare.db


In [5]:
import sqlite3
import pandas as pd

os.chdir("/Users/adnan/Desktop/healthcare-bi-portfolio")
conn = sqlite3.connect("data/healthcare.db")

queries = {

"sql/01_state_avg_score.sql": """
SELECT
    state,
    COUNT(DISTINCT facility_id)         AS hospital_count,
    ROUND(AVG(score), 2)                AS avg_score,
    ROUND(AVG(lower_estimate), 2)       AS avg_lower,
    ROUND(AVG(higher_estimate), 2)      AS avg_higher
FROM hospital_measures
GROUP BY state
ORDER BY avg_score DESC
""",

"sql/02_top_measures.sql": """
SELECT
    measure_id,
    measure_name,
    COUNT(*)                            AS hospital_count,
    ROUND(AVG(score), 2)               AS avg_score
FROM hospital_measures
GROUP BY measure_id, measure_name
ORDER BY avg_score DESC
LIMIT 15
""",

"sql/03_hospital_rankings.sql": """
SELECT
    facility_id,
    facility_name,
    city,
    state,
    COUNT(DISTINCT measure_id)          AS measures_tracked,
    ROUND(AVG(score), 2)               AS avg_score
FROM hospital_measures
GROUP BY facility_id, facility_name, city, state
HAVING measures_tracked >= 5
ORDER BY avg_score ASC
LIMIT 50
""",

"sql/04_compared_to_national.sql": """
SELECT
    compared_to_national,
    COUNT(*)                            AS record_count,
    ROUND(AVG(score), 2)               AS avg_score
FROM hospital_measures
GROUP BY compared_to_national
ORDER BY record_count DESC
"""

}

# Write SQL files and preview results
for filepath, sql in queries.items():
    with open(filepath, "w") as f:
        f.write(sql.strip())
    print(f"\n{'='*55}")
    print(f" {filepath}")
    print('='*55)
    print(pd.read_sql_query(sql, conn).to_string(index=False))

# Export clean CSVs for Looker Studio
exports = {
    "data/clean/state_avg_score.csv":        "sql/01_state_avg_score.sql",
    "data/clean/top_measures.csv":           "sql/02_top_measures.sql",
    "data/clean/hospital_rankings.csv":      "sql/03_hospital_rankings.sql",
    "data/clean/compared_to_national.csv":   "sql/04_compared_to_national.sql",
}

for csv_path, sql_path in exports.items():
    with open(sql_path) as f:
        sql = f.read()
    pd.read_sql_query(sql, conn).to_csv(csv_path, index=False)
    print(f"Exported: {csv_path}")

conn.close()
print("\nPhase 2 complete.")


 sql/01_state_avg_score.sql
state  hospital_count  avg_score  avg_lower  avg_higher
   MP               1      16.52      11.32       23.70
   VI               2      14.03       9.55       20.29
   DC               7      13.25       9.35       17.45
   NJ              62      12.68       8.58       17.04
   AK              16      12.63       8.60       17.19
   MD              43      12.56       8.57       16.80
   WA              78      12.45       8.58       16.71
   NV              31      12.36       8.34       16.77
   MT              36      12.16       8.20       16.69
   ND              33      12.13       8.20       16.76
   CT              27      11.84       7.91       16.06
   NH              26      11.79       8.02       16.05
   MO              97      11.55       7.77       15.73
   GU               1      11.47       7.87       16.37
   GA             111      11.28       7.48       15.43
   NY             147      11.23       7.60       15.16
   DE              

Exported: data/clean/state_avg_score.csv
Exported: data/clean/top_measures.csv
Exported: data/clean/hospital_rankings.csv
Exported: data/clean/compared_to_national.csv

Phase 2 complete.


In [2]:
readme = """# Healthcare Operations BI Dashboard

An end-to-end business intelligence project analyzing U.S. hospital complication 
and death rates using CMS public data, Python, SQL, and Looker Studio.

## Live Dashboard
[View on Looker Studio](YOUR_LINK_HERE)

## Problem Statement
Hospital complication and mortality rates vary significantly across the U.S. 
This project identifies which states, hospitals, and measure types show the 
highest and lowest rates — giving healthcare administrators and policymakers 
a data-driven starting point for quality improvement.

## Dataset
- **Source:** CMS Hospital Compare — Complications & Deaths
- **Link:** https://data.cms.gov/provider-data/dataset/ynj2-r877
- **Size:** 95,780 rows × 18 columns (raw), 50,478 rows after cleaning
- **Period:** Most recent reporting period (filtered to latest end date)

## Tools & Stack
| Layer        | Tool                        |
|--------------|-----------------------------|
| Cleaning     | Python (pandas)             |
| Storage      | SQLite                      |
| Analysis     | SQL                         |
| Dashboard    | Looker Studio               |
| Version Control | Git / GitHub             |

## Project Structure
healthcare-bi-portfolio/
├── data/
│   ├── raw/          ← original CMS CSV (not tracked in git)
│   └── clean/        ← processed CSVs for Looker Studio
├── sql/              ← analytical SQL queries
├── notebooks/        ← Python cleaning scripts
├── outputs/          ← screenshots and exports
└── README.md

## Key Findings
1. **Hospitals rated worse than national average score 2x higher** than 
   those rated better (22.36 vs 11.42 avg score)
2. **PSI-04 (surgical death rate)** has an avg score of 173.90 — measured 
   per 1,000 patients, making it the highest-magnitude metric in the dataset
3. **Specialty orthopedic hospitals dominate the best-performer rankings** 
   likely due to elective, lower-risk procedure mix
4. **Mariana Islands and Virgin Islands** show elevated scores but represent 
   1-2 hospitals — small sample size limits conclusions

## Data Dictionary
| Column | Description |
|--------|-------------|
| facility_id | Unique CMS hospital identifier |
| facility_name | Hospital name |
| city, state | Location |
| measure_id | CMS measure code (e.g. MORT_30_PN) |
| measure_name | Human-readable measure description |
| compared_to_national | Performance vs national benchmark |
| score | Rate per 1,000 patients (scale varies by measure) |
| lower_estimate / higher_estimate | 95% confidence interval |

## Methodology
1. Downloaded raw CMS CSV and loaded into pandas
2. Dropped non-analytical columns (Footnote, Address, Telephone)
3. Cast numeric columns and parsed dates
4. Filtered to most recent reporting period only
5. Removed records with null scores
6. Loaded into SQLite and wrote 4 analytical queries
7. Exported clean CSVs to Looker Studio via Google Sheets
"""

with open("README.md", "w") as f:
    f.write(readme)

print("README.md written.")

README.md written.
